In [ ]:
#This script investigated the viability of the Heston model for modeling intraday market volatility
#It also creates a data pipeline for a resulting Heston environment but was discarded due to non convergent bahaviour
!pip install arch
!pip install entsoe-py

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from sklearn.preprocessing import StandardScaler
from scipy.optimize import minimize
from entsoe import EntsoePandasClient

# 1. Load data

API_KEY    = "[REDACTED]"
DRIVE_PATH = "/content/drive/MyDrive/MEng Project"

da = pd.read_csv(f"{DRIVE_PATH}/datasdayahead_prices.csv", parse_dates=["timestamp"])
sm = pd.read_csv(f"{DRIVE_PATH}/datassmard_fundamentals.csv", parse_dates=["timestamp"])

da = da.drop(columns=["market"], errors="ignore")
sm = sm.drop(columns=["wind_offshore_mw"], errors="ignore")

df = da.merge(
    sm[["timestamp", "wind_total_mw", "load_mw"]],
    on="timestamp", how="inner"
).sort_values("timestamp").reset_index(drop=True)

# 1b. Download Wind Forecast Error

print("=" * 55)
print("DOWNLOADING WIND FORECAST (ENTSO-E A69)")
print("=" * 55)

try:
    client = EntsoePandasClient(api_key=API_KEY)

    start = pd.Timestamp("2022-10-01", tz="UTC")
    end   = pd.Timestamp("2025-03-01", tz="UTC")

    wind_forecast = client.query_wind_and_solar_forecast(
        "10Y1001A1001A82H",
        start=start,
        end=end,
        psr_type="B19"
    )

    wind_forecast_on = client.query_wind_and_solar_forecast(
        "10Y1001A1001A82H",
        start=start,
        end=end,
        psr_type="B18"
    )

    wf = pd.DataFrame({
        "wind_forecast_mw": wind_forecast.squeeze() + wind_forecast_on.squeeze()
    }).reset_index()

    wf.columns = ["timestamp", "wind_forecast_mw"]
    wf["timestamp"] = pd.to_datetime(wf["timestamp"], utc=True)

    wf = wf.set_index("timestamp").resample("15min").ffill().reset_index()

    df = df.merge(wf, on="timestamp", how="left")

    df["wfe_mw"] = df["wind_forecast_mw"] - df["wind_total_mw"]

    print(f"  WFE rows merged: {df['wfe_mw'].notna().sum()}")
    print(f"  WFE mean: {df['wfe_mw'].mean():.1f} MW")
    print(f"  WFE std:  {df['wfe_mw'].std():.1f} MW")

except Exception as e:
    print(f"  WARNING: Could not download wind forecast: {e}")
    print("  Setting WFE to zero — re-run when API is available")
    df["wind_forecast_mw"] = np.nan
    df["wfe_mw"] = 0.0

# 2. Compute log returns

df["log_return"] = np.log(
    df["price_eur_mwh"].clip(lower=1) /
    df["price_eur_mwh"].shift(1).clip(lower=1)
) * 100

df = df.dropna(subset=["log_return"]).reset_index(drop=True)

train_mask = df["season"].isin(["2022/23", "2023/24"])
train = df[train_mask].copy()

# 3. Calibrate Heston model

print("=" * 55)
print("CALIBRATING HESTON MODEL (training data only)")
print("=" * 55)

RHO = -0.7
DT  = 1.0

def simulate_heston_variance(log_returns, kappa, theta, xi, V0, rho, dt=1.0):
    """
    Discretised Heston variance path using Euler scheme.
    """
    n = len(log_returns)
    V = np.zeros(n)
    V[0] = V0

    returns_std = log_returns.values / (np.std(log_returns.values) + 1e-8)

    for t in range(1, n):
        dW1 = returns_std[t]
        dW2 = rho * dW1 + np.sqrt(1 - rho**2) * np.random.normal()
        V[t] = (
            V[t-1]
            + kappa * (theta - V[t-1]) * dt
            + xi * np.sqrt(max(V[t-1], 1e-8)) * dW2 * np.sqrt(dt)
        )
        V[t] = max(V[t], 1e-8)

    return V

def heston_neg_loglik(params, log_returns):
    kappa, theta, xi = params

    if kappa <= 0 or theta <= 0 or xi <= 0:
        return 1e10
    if 2 * kappa * theta <= xi**2:
        return 1e10

    V0 = np.var(log_returns.values)
    np.random.seed(42)
    V = simulate_heston_variance(log_returns, kappa, theta, xi, V0, RHO, DT)

    sigma_t = np.sqrt(np.maximum(V, 1e-8))
    ll = -0.5 * np.sum(
        np.log(2 * np.pi * sigma_t**2) + (log_returns.values**2) / sigma_t**2
    )

    return -ll

x0 = [0.5, np.var(train["log_return"].values), 0.3]
bounds = [(0.01, 10.0), (0.001, 100.0), (0.01, 5.0)]

result = minimize(
    heston_neg_loglik,
    x0,
    args=(train["log_return"],),
    method="L-BFGS-B",
    bounds=bounds,
    options={"maxiter": 500, "ftol": 1e-9}
)

kappa_fit, theta_fit, xi_fit = result.x
V0_fit = np.var(train["log_return"].values)

print(f"  Calibration success: {result.success}")
print(f"  kappa: {kappa_fit:.4f}")
print(f"  theta: {theta_fit:.4f}")
print(f"  xi:    {xi_fit:.4f}")
print(f"  V0:    {V0_fit:.4f}")
print(f"  rho:   {RHO}")
print(f"  Feller: {2*kappa_fit*theta_fit:.4f} > {xi_fit**2:.4f}")

# 4. Simulate volatility

SIGMA_FLOOR = 0.5
SIGMA_CAP   = 30.0

np.random.seed(42)
V_full = simulate_heston_variance(
    df["log_return"],
    kappa_fit, theta_fit, xi_fit, V0_fit, RHO, DT
)

mean_price = df.loc[train_mask, "price_eur_mwh"].mean()

df["sigma"] = (np.sqrt(V_full) * mean_price / 100).clip(SIGMA_FLOOR, SIGMA_CAP)

print("\nsigma stats:")
print(df.groupby("season")["sigma"].describe().round(3))
print()

# 5. OU process

def simulate_ou(sigma_series, theta):
    n = len(sigma_series)
    epsilon = np.zeros(n)

    for t in range(1, n):
        epsilon[t] = (
            epsilon[t-1]
            + theta * (0 - epsilon[t-1])
            + sigma_series[t] * np.random.normal()
        )

    return epsilon

df["epsilon"] = simulate_ou(df["sigma"].values, theta=0.1)
df["p_intraday"] = df["price_eur_mwh"] + df["epsilon"]

# 6. Time to delivery

df["ttd_min"] = 60 - df["timestamp"].dt.minute
df["ttd_norm"] = df["ttd_min"] / 60.0

# 7. Normalisation

def normalise(col, df, mask):
    mu = df.loc[mask, col].mean()
    std = df.loc[mask, col].std()
    return (df[col] - mu) / std, mu, std

df["wfe_norm"], mu_wfe, std_wfe = normalise("wfe_mw", df, train_mask)

# 8. Save

# Create state dataset (what the agent actually sees)

STATE_COLS = [
    "timestamp",
    "season",
    "price_eur_mwh",
    "p_intraday",
    "epsilon",
    "sigma",
    "wind_total_mw",
    "load_mw",
    "wfe_mw",
    "ttd_norm"
]

state_df = df[STATE_COLS].copy()

# Train / test split

train_df = state_df[state_df["season"].isin(["2022/23", "2023/24"])].reset_index(drop=True)
test_df  = state_df[state_df["season"] == "2024/25"].reset_index(drop=True)

train_df.to_csv(f"{DRIVE_PATH}/heston_train.csv", index=False)
test_df.to_csv(f"{DRIVE_PATH}/heston_test.csv", index=False)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.3/981.3 kB 15.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 15.0 MB/s eta 0:00:00
DOWNLOADING WIND FORECAST (ENTSO-E A69)
  WFE rows merged: 23408
  WFE mean: 14650.7 MW
  WFE std:  9407.3 MW
CALIBRATING HESTON MODEL (training data only)
  Calibration success: True
  kappa: 0.0100
  theta: 100.0000
  xi:    0.0100
  V0:    8911.9975
  rho:   -0.7
  Feller: 2.0000 > 0.0001

sigma stats:
          count    mean    std     min     25%     50%     75%     max
season                                                                
2022/23  7748.0  13.616  3.612  12.564  12.665  12.687  12.717  30.000
2023/24  7854.0  12.690  0.029  12.605  12.671  12.691  12.710  12.785
2024/25  7821.0  12.687  0.032  12.568  12.667  12.691  12.710  12.777

